# Link5Dots APK Builder
Run cells 1-6 top to bottom. First build ~30-45 min. Cached builds ~5 min.

| Step | What | Time |
|------|------|------|
| 1 | System packages | ~30s |
| 2 | Python tools | ~30s |
| 3 | Clone repo | ~10s |
| 4 | Patch p4a (fixes Python 3.14 pip bug) | ~30s |
| 5 | Build APK | ~30-45 min first run |
| 6 | Download APK | instant |

In [ ]:
# Step 1: System Dependencies
%%bash
sudo add-apt-repository ppa:deadsnakes/ppa -y 2>/dev/null
sudo apt-get update -qq
sudo apt-get install -y --no-install-recommends \
    git zip unzip wget \
    build-essential libssl-dev libffi-dev python3.11-dev python3.11-venv \
    openjdk-17-jdk-headless \
    autoconf libtool pkg-config cmake \
    libltdl-dev libxml2-dev libxslt1-dev \
    zlib1g-dev lld ccache \
    2>/dev/null
echo "System dependencies installed"

In [ ]:
# Step 2: Python Build Tools (using Python 3.11 to avoid 3.14 API bugs)
import urllib.request
urllib.request.urlretrieve('https://bootstrap.pypa.io/get-pip.py', 'get-pip.py')
!python3.11 get-pip.py -q
!python3.11 -m pip install -q setuptools==70.3.0 wheel==0.43.0
!python3.11 -m pip install -q "Cython==0.29.36"
!python3.11 -m pip install -q buildozer==1.5.0
print('Build tools installed')
!python3.11 -m buildozer --version

In [ ]:
# Step 3: Clone / Update Repo
import os

REPO_URL = 'https://github.com/Mufaiz-std/link5dots.git'
REPO_DIR = '/content/link5dots'

if os.path.exists(REPO_DIR):
    print('Repo exists, pulling latest...')
    !cd /content/link5dots && git pull
else:
    print('Cloning repo...')
    !git clone $REPO_URL $REPO_DIR

os.chdir(REPO_DIR)
print('Repo contents:')
!ls

In [ ]:
# Step 4: Clear old caches
import os, shutil
venv = '/content/link5dots/.buildozer/android/platform/build-arm64-v8a_armeabi-v7a/build/venv'
if os.path.exists(venv):
    shutil.rmtree(venv)
print('Ready to build!')


In [ ]:
# Step 5: Build APK (first run ~30-45 min, cached ~5 min)
import os
os.chdir('/content/link5dots')
print('Building APK... grab a coffee')
print('Watch for: [INFO]: APK ... is ready')
!python3.11 -m buildozer android debug 2>&1

In [ ]:
# Step 6: Download APK
import glob, os
from google.colab import files

apks = glob.glob('/content/link5dots/bin/*.apk')
if apks:
    apk = apks[0]
    mb = os.path.getsize(apk) / 1024 / 1024
    print(f'APK ready! {os.path.basename(apk)} ({mb:.1f} MB)')
    files.download(apk)
else:
    print('No APK found - check build output above for errors')

## Optional: Cache to Google Drive
Run Optional-A after first successful build to save the cache.
Run Optional-B at the start of a new session to restore (future builds ~5 min).

In [ ]:
# Optional-A: SAVE cache to Drive (run after successful build)
from google.colab import drive
import os
drive.mount('/content/drive')
CACHE = '/content/drive/MyDrive/link5dots_buildcache'
os.makedirs(CACHE, exist_ok=True)
print('Saving ~/.buildozer to Drive... (2-3 min)')
!tar czf $CACHE/buildozer_cache.tar.gz -C /root .buildozer
print('Saving project .buildozer...')
!tar czf $CACHE/project_buildozer.tar.gz -C /content/link5dots .buildozer
print('Cache saved to Google Drive!')

In [ ]:
# Optional-B: RESTORE cache from Drive (run BEFORE Step 5 in new session)
from google.colab import drive
import os
drive.mount('/content/drive')
CACHE = '/content/drive/MyDrive/link5dots_buildcache'
if os.path.exists(f'{CACHE}/buildozer_cache.tar.gz'):
    print('Restoring ~/.buildozer...')
    !tar xzf $CACHE/buildozer_cache.tar.gz -C /root
    print('Restored ~/.buildozer')
if os.path.exists(f'{CACHE}/project_buildozer.tar.gz'):
    print('Restoring project cache...')
    os.makedirs('/content/link5dots', exist_ok=True)
    !tar xzf $CACHE/project_buildozer.tar.gz -C /content/link5dots
    print('Restored project cache')
print('Done! Now run Steps 1-5. Build will take ~5 min.')